<a href="https://colab.research.google.com/github/sl-Dubbing/dubbing-studio/blob/main/sl_dubbing_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ sl-Dubbing TTS Server
## Bark (Suno) + ngrok = API مجاني بـ GPU

### خطوات التشغيل:
1. شغّل كل الخلايا بالترتيب
2. انسخ رابط ngrok
3. ضعه في `dubbing.html` كـ `BE`

In [1]:
!pip install -q coqui-tts flask flask-cors pyngrok requests gtts
!apt-get install -qq ffmpeg
print('✅ Done')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.8/862.8 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.3/997.3 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 648.4/648.4 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 1.0 MB/s eta 0:00:00
✅ Done


In [2]:
# إصلاح torch.load
import torch
import builtins

# استعادة torch.load الأصلي
if hasattr(torch, '_orig_load'):
    torch.load = torch._orig_load
else:
    torch._orig_load = torch.load

print('✅ torch.load fixed')

✅ torch.load fixed


In [3]:
import os
os.environ['COQUI_TOS_AGREED'] = '1'
import torch
from TTS.api import TTS

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)
print('✅ XTTS v2 ready!')

Device: cpu


100%|██████████| 1.87G/1.87G [00:39<00:00, 47.7MiB/s]
4.37kiB [00:00, 4.75MiB/s]
361kiB [00:00, 74.6MiB/s]
100%|██████████| 32.0/32.0 [00:00<00:00, 48.4kiB/s]
100%|██████████| 7.75M/7.75M [00:00<00:00, 8.86MiB/s]


✅ XTTS v2 ready!


In [4]:
import requests, os
from pathlib import Path

CLOUD_NAME    = 'dxbmvzsiz'
DEFAULT_VOICE = '5_gtygjb'
FOLDER        = 'sl_voices'
voice_path    = Path('/tmp/voice_sample.wav')

def fetch_voice():
    if voice_path.exists() and voice_path.stat().st_size > 5000:
        print(f'✅ Voice cached: {voice_path.stat().st_size/1024:.1f}KB')
        return str(voice_path)
    urls = [
        f'https://res.cloudinary.com/{CLOUD_NAME}/raw/upload/{FOLDER}/{DEFAULT_VOICE}',
        f'https://res.cloudinary.com/{CLOUD_NAME}/video/upload/v1773450710/{DEFAULT_VOICE}.mp3',
    ]
    for url in urls:
        r = requests.get(url, timeout=30)
        if r.status_code == 200:
            ext = '.mp3' if '.mp3' in url else '.wav'
            tmp = Path(f'/tmp/voice{ext}')
            tmp.write_bytes(r.content)
            if ext == '.mp3':
                os.system(f'ffmpeg -y -i {tmp} -ar 22050 -ac 1 {voice_path}')
                tmp.unlink()
            else:
                import shutil; shutil.copy(str(tmp), str(voice_path))
            print(f'✅ Voice ready: {voice_path.stat().st_size/1024:.1f}KB')
            return str(voice_path)
    print('⚠️ Voice not found')
    return None

VOICE_FILE = fetch_voice()

✅ Voice ready: 201.9KB


In [5]:
# ── تحميل srt_sync ──
with open('/tmp/srt_sync.py', 'w', encoding='utf-8') as f:
    f.write('''
import re, uuid, wave, subprocess, logging
from pathlib import Path

logger = logging.getLogger(__name__)
AUDIO_DIR = Path('/tmp/sl_audio')
AUDIO_DIR.mkdir(exist_ok=True)

def parse_srt(content):
    entries = []
    blocks = re.split(r"\\n\\s*\\n", content.strip())
    for block in blocks:
        lines = block.strip().split("\\n")
        if len(lines) < 3 or not lines[0].strip().isdigit(): continue
        m = re.match(r"(\\d{2}):(\\d{2}):(\\d{2})[,.]( \\d{3})\\s*-->\\s*(\\d{2}):(\\d{2}):(\\d{2})[,.](\\d{3})", lines[1].strip())
        if not m: continue
        h1,m1,s1,ms1,h2,m2,s2,ms2 = map(int, m.groups())
        start_ms = (h1*3600+m1*60+s1)*1000+ms1
        end_ms = (h2*3600+m2*60+s2)*1000+ms2
        text = " ".join(lines[2:]).strip()
        if text:
            entries.append({"index":int(lines[0]),"start_ms":start_ms,"end_ms":end_ms,"duration_ms":end_ms-start_ms,"text":text})
    return entries

def get_dur_ms(path):
    try:
        r = subprocess.run(["ffprobe","-v","error","-show_entries","format=duration","-of","csv=p=0",path], capture_output=True, text=True, timeout=10)
        return float(r.stdout.strip()) * 1000
    except: return 0

def fit_audio(inp, out, target_ms):
    orig = get_dur_ms(inp)
    if orig <= 0: return False
    ratio = orig / target_ms
    if ratio < 0.5: at = f"atempo=0.5,atempo={ratio/0.5:.4f}"
    elif ratio > 2: at = f"atempo=2.0,atempo={ratio/2.0:.4f}"
    else: at = f"atempo={ratio:.4f}"
    subprocess.run(["ffmpeg","-y","-i",inp,"-filter:a",at,out], capture_output=True, timeout=30)
    return Path(out).exists() and Path(out).stat().st_size > 100

def merge_with_timing(segments, total_ms, output, sr=22050):
    try:
        buf = bytearray(int(sr * total_ms / 1000) * 2)
        for seg in segments:
            wp = seg["wav_path"]
            if not Path(wp).exists(): continue
            if not wp.endswith(".wav"):
                c = wp + ".wav"
                subprocess.run(["ffmpeg","-y","-i",wp,"-ar",str(sr),"-ac","1",c], capture_output=True, timeout=20)
                wp = c if Path(c).exists() else wp
            try:
                with wave.open(wp,"rb") as w: frames = w.readframes(w.getnframes())
            except: continue
            sb = int(sr * seg["start_ms"] / 1000) * 2
            eb = min(sb + len(frames), len(buf))
            if eb - sb > 0: buf[sb:eb] = frames[:eb-sb]
        with wave.open(output,"wb") as o:
            o.setnchannels(1); o.setsampwidth(2); o.setframerate(sr)
            o.writeframes(bytes(buf))
        return True
    except Exception as e:
        print(f"merge error: {e}"); return False

def dub_srt(srt_content, lang, tts_func, voice_cache=None):
    entries = parse_srt(srt_content)
    if not entries: return None
    total_ms = entries[-1]["end_ms"] + 1000
    segments, tmp = [], []
    for e in entries:
        raw = tts_func(e["text"], lang)
        if not raw or not Path(raw).exists(): continue
        tmp.append(raw)
        fitted = str(AUDIO_DIR / f"fit_{uuid.uuid4()}.wav")
        if fit_audio(raw, fitted, e["duration_ms"]):
            segments.append({"start_ms": e["start_ms"], "wav_path": fitted})
            tmp.append(fitted)
        else:
            segments.append({"start_ms": e["start_ms"], "wav_path": raw})
    if not segments: return None
    out = str(AUDIO_DIR / f"dubbed_{uuid.uuid4()}.wav")
    ok = merge_with_timing(segments, total_ms, out)
    for f in tmp:
        try: Path(f).unlink(missing_ok=True)
        except: pass
    return out if ok else None
''')

import sys
sys.path.insert(0, '/tmp')
from srt_sync import dub_srt, parse_srt
print('✅ srt_sync جاهز!')

✅ srt_sync جاهز!


In [6]:
import uuid, threading, time, subprocess
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS
from pathlib import Path

app = Flask(__name__)
CORS(app, origins=['*'])
AUDIO_DIR = Path('/tmp/sl_audio')
AUDIO_DIR.mkdir(exist_ok=True)

XTTS_LANGS = {
    'ar':'ar','en':'en','es':'es','fr':'fr','de':'de',
    'it':'it','ru':'ru','tr':'tr','zh':'zh-cn','hi':'hi','nl':'nl'
}
GTTS_LANGS = {
    'ar':'ar','en':'en','es':'es','fr':'fr','de':'de','it':'it',
    'ru':'ru','tr':'tr','zh':'zh-TW','hi':'hi','fa':'fa','sv':'sv','nl':'nl'
}

# تقسيم النص حسب عدد الرموز (tokens)
def split_text_by_tokens(text, max_tokens=380):
    import re
    sentences = re.split(r'[.؟!،]', text)
    chunks, cur = [], ""
    for s in sentences:
        s = s.strip()
        if not s: continue
        if len(cur.split()) + len(s.split()) < max_tokens:
            cur += s + ". "
        else:
            if cur: chunks.append(cur.strip())
            cur = s + ". "
    if cur: chunks.append(cur.strip())
    return chunks or [text]

def gen_xtts(text, lang):
    xl  = XTTS_LANGS.get(lang, 'en')
    chunks = split_text_by_tokens(text)
    out_files = []
    for chunk in chunks:
        out = str(AUDIO_DIR / f'xtts_{uuid.uuid4()}.wav')
        if VOICE_CACHE:
            tts.synthesizer.tts_model.inference(
                text=chunk, language=xl,
                gpt_cond_latent=VOICE_CACHE['gpt_cond_latent'],
                speaker_embedding=VOICE_CACHE['speaker_embedding'],
                output_path=out
            )
        else:
            tts.tts_to_file(text=chunk, speaker_wav=str(voice_path),
                            language=xl, file_path=out)
        out_files.append(out)
    # دمج الملفات إذا كان أكثر من جزء
    if len(out_files) == 1:
        return out_files[0]
    merged = str(AUDIO_DIR / f'xtts_{uuid.uuid4()}_merged.wav')
    import wave
    params, data = None, []
    for f in out_files:
        with wave.open(f, 'rb') as w:
            if not params: params = w.getparams()
            data.append(w.readframes(w.getnframes()))
    with wave.open(merged, 'wb') as out:
        out.setparams(params)
        for d in data: out.writeframes(d)
    return merged

def gen_gtts(text, lang):
    from gtts import gTTS
    out = str(AUDIO_DIR / f'gtts_{uuid.uuid4()}.mp3')
    GTS_L = GTTS_LANGS.get(lang,'en')
    gTTS(text=text, lang=GTS_L).save(out)
    # تحويل MP3 إلى WAV للدمج
    wav = out.replace('.mp3','.wav')
    subprocess.run(['ffmpeg','-y','-i',out,'-ar','22050','-ac','1',wav],
                   capture_output=True, timeout=30)
    Path(out).unlink(missing_ok=True)
    return wav if Path(wav).exists() else out

@app.route('/')
@app.route('/api/health')
def health():
    return jsonify({
        'status':      'ok',
        'service':     'sl-Dubbing Colab',
        'gpu':         torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
        'voice_ready': voice_path.exists(),
        'cache_ready': VOICE_CACHE is not None
    })

@app.route('/api/dub', methods=['POST'])
def dub():
    try:
        d          = request.get_json() or {}
        text       = d.get('text','').strip()
        lang       = d.get('lang','ar')
        mode       = d.get('voice_mode','gtts')
        if not text: return jsonify({'error':'النص فارغ'}), 400
        if mode=='xtts' and XTTS_LANGS.get(lang) and voice_path.exists():
            path = gen_xtts(text, lang); method = 'xtts_v2'
        else:
            path = gen_gtts(text, lang); method = 'gtts'
        fname = Path(path).name
        url = f"{request.host_url.rstrip('/')}/api/download/{fname}"
        return jsonify({'success':True,'audio_url':url,'method':method,'lang':lang})
    except Exception as e:
        return jsonify({'error':str(e)}), 500

@app.route('/api/download/<filename>')
def download(filename):
    p = AUDIO_DIR / Path(filename).name
    mime = 'audio/wav' if str(p).endswith('.wav') else 'audio/mpeg'
    return send_file(str(p), mimetype=mime, as_attachment=True)

print('✅ Flask جاهز')

✅ Flask جاهز


In [7]:
# ── الخلية 6: تشغيل + ngrok ──
NGROK_TOKEN = '1vBJLYeOJQxiZO65zWISXZHTvFh_2nbx3jLyyo5sAFKvXWoXn'

from pyngrok import ngrok
import threading, time

ngrok.set_auth_token(NGROK_TOKEN)

PORT = 5050  # يمكنك تغيير المنفذ إذا كان 5000 مستخدم

t = threading.Thread(target=lambda: app.run(port=PORT, use_reloader=False))
t.daemon = True
t.start()
time.sleep(2)

tunnel = ngrok.connect(PORT)
PUBLIC_URL = tunnel.public_url

print('=' * 55)
print('🚀 الخادم يعمل!')
print(f'🌐 {PUBLIC_URL}')
print('=' * 55)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5050
INFO:werkzeug:Press CTRL+C to quit


🚀 الخادم يعمل!
🌐 https://f517-35-185-133-229.ngrok-free.app


In [8]:
# ═══════════════════════════════════════════════════════
# أضف هذا في نهاية الخلية 6 (بعد ngrok)
# يحفظ الرابط تلقائياً في Cloudinary
# ═══════════════════════════════════════════════════════
import requests, hashlib, time, json

CLOUD_NAME = 'dxbmvzsiz'
API_KEY    = '432687952743126'
API_SECRET = 'BrFvzlPFXBJZ-B-cZyxCc-0wHRo'

def save_url_to_cloudinary(url):
    """يحفظ رابط ngrok في Cloudinary كملف JSON"""
    timestamp = int(time.time())
    data = json.dumps({'url': url, 'ts': timestamp}).encode()

    # رفع كـ raw file
    sig_str = f"folder=config&public_id=backend_url&timestamp={timestamp}{API_SECRET}"
    signature = hashlib.sha1(sig_str.encode()).hexdigest()

    res = requests.post(
        f"https://api.cloudinary.com/v1_1/{CLOUD_NAME}/raw/upload",
        data={
            'api_key':   API_KEY,
            'timestamp': timestamp,
            'signature': signature,
            'folder':    'config',
            'public_id': 'backend_url',
        },
        files={'file': ('backend_url.json', data, 'application/json')},
        timeout=30
    )
    if res.status_code == 200:
        print(f"✅ الرابط محفوظ في Cloudinary!")
        return True
    else:
        print(f"❌ فشل الحفظ: {res.text[:100]}")
        return False

# احفظ الرابط تلقائياً
save_url_to_cloudinary(PUBLIC_URL)
print(f"🌐 الرابط الحالي: {PUBLIC_URL}")

✅ الرابط محفوظ في Cloudinary!
🌐 الرابط الحالي: https://f517-35-185-133-229.ngrok-free.app


In [9]:
import pickle, hashlib, time
from pathlib import Path

CACHE_DIR  = Path('/tmp/voice_cache')
CACHE_DIR.mkdir(exist_ok=True)
CACHE_FILE = CACHE_DIR / 'abdulselam_embedding.pkl'

def build_cache():
    if not voice_path.exists():
        print("❌ ملف الصوت غير موجود")
        return None
    if CACHE_FILE.exists():
        with open(CACHE_FILE, 'rb') as f:
            cache = pickle.load(f)
        print(f"✅ Cache موجود — عمره: {(time.time()-cache['ts'])/3600:.1f} ساعة")
        return cache
    print("⏳ جاري استخراج النبرة... (مرة واحدة فقط)")
    start = time.time()
    gpt_cond_latent, speaker_embedding = tts.synthesizer.tts_model.get_conditioning_latents(
        audio_path=[str(voice_path)]
    )
    cache = {
        'gpt_cond_latent':   gpt_cond_latent,
        'speaker_embedding': speaker_embedding,
        'ts': time.time()
    }
    with open(CACHE_FILE, 'wb') as f:
        pickle.dump(cache, f)
    print(f"✅ تم بناء الـ Cache في {time.time()-start:.1f} ثانية!")
    return cache

VOICE_CACHE = build_cache()
print("✅ Voice Cache جاهز!")

⏳ جاري استخراج النبرة... (مرة واحدة فقط)
✅ تم بناء الـ Cache في 2.1 ثانية!
✅ Voice Cache جاهز!


In [10]:
import uuid, threading, time
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS
from pathlib import Path
import torch

app = Flask(__name__)
CORS(app, resources={r"/*": {"origins": "*"}}, supports_credentials=True)

AUDIO_DIR = Path('/tmp/sl_audio')
AUDIO_DIR.mkdir(exist_ok=True)

XTTS_LANGS = {
    'ar':'ar','en':'en','es':'es','fr':'fr','de':'de',
    'it':'it','ru':'ru','tr':'tr','zh':'zh-cn','hi':'hi','nl':'nl'
}
GTTS_LANGS = {
    'ar':'ar','en':'en','es':'es','fr':'fr','de':'de','it':'it',
    'ru':'ru','tr':'tr','zh':'zh-TW','hi':'hi','fa':'fa','sv':'sv','nl':'nl'
}

def gen_xtts(text, lang):
    xl  = XTTS_LANGS.get(lang)
    out = str(AUDIO_DIR / f'xtts_{uuid.uuid4()}.wav')
    tts.tts_to_file(text=text, speaker_wav=str(voice_path),
                    language=xl, file_path=out)
    return out

def gen_gtts(text, lang):
    from gtts import gTTS
    out = str(AUDIO_DIR / f'gtts_{uuid.uuid4()}.mp3')
    gTTS(text=text, lang=GTTS_LANGS.get(lang,'en')).save(out)
    return out

@app.route('/')
@app.route('/api/health')
def health():
    return jsonify({
        'status':'ok',
        'service':'sl-Dubbing Colab',
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
        'voice_ready': voice_path.exists()
    })

@app.route('/api/dub', methods=['POST'])
def dub():
    try:
        d    = request.get_json() or {}
        text = d.get('text','').strip()
        lang = d.get('lang','ar')
        mode = d.get('voice_mode','gtts')
        if not text:
            return jsonify({'error':'النص فارغ'}), 400
        if mode == 'xtts' and XTTS_LANGS.get(lang) and voice_path.exists():
            path   = gen_xtts(text, lang)
            method = 'xtts_v2'
        else:
            path   = gen_gtts(text, lang)
            method = 'gtts'
        fname = Path(path).name
        url   = f"{request.host_url.rstrip('/')}/api/download/{fname}"
        return jsonify({'success':True,'audio_url':url,'method':method})
    except Exception as e:
        return jsonify({'error':str(e)}), 500

@app.route('/api/download/<filename>')
def download(filename):
    p    = AUDIO_DIR / Path(filename).name
    mime = 'audio/wav' if str(p).endswith('.wav') else 'audio/mpeg'
    return send_file(str(p), mimetype=mime, as_attachment=True)

print('✅ Flask defined')

✅ Flask defined


In [11]:
# ضع token هنا من: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = '1vBJLYeOJQxiZO65zWISXZHTvFh_2nbx3jLyyo5sAFKvXWoXn'

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)

t = threading.Thread(target=lambda: app.run(port=5000, use_reloader=False))
t.daemon = True
t.start()
time.sleep(2)

tunnel = ngrok.connect(5000)
PUBLIC_URL = tunnel.public_url

print('=' * 55)
print('🚀 الخادم يعمل!')
print(f'🌐 {PUBLIC_URL}')
print('=' * 55)
print(f"\nconst BE = '{PUBLIC_URL}';")
print('=' * 55)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


🚀 الخادم يعمل!
🌐 https://28a7-35-185-133-229.ngrok-free.app

const BE = 'https://28a7-35-185-133-229.ngrok-free.app';


In [12]:
import requests

r = requests.get(f'{PUBLIC_URL}/api/health')
print('Health:', r.json())

r = requests.post(f'{PUBLIC_URL}/api/dub', json={
    'text': 'مرحبا هذا اختبار',
    'lang': 'ar',
    'voice_mode': 'gtts'
})
print('Dub test:', r.json())

INFO:werkzeug:127.0.0.1 - - [15/Mar/2026 14:34:46] "GET /api/health HTTP/1.1" 200 -


Health: {'gpu': 'CPU', 'service': 'sl-Dubbing Colab', 'status': 'ok', 'voice_ready': True}


INFO:werkzeug:127.0.0.1 - - [15/Mar/2026 14:34:47] "POST /api/dub HTTP/1.1" 200 -


Dub test: {'audio_url': 'http://28a7-35-185-133-229.ngrok-free.app/api/download/gtts_f0d69890-33cf-4330-a6d0-5104540330b2.mp3', 'method': 'gtts', 'success': True}


In [13]:
# ═══════════════════════════════════════════
# الخلية 7 — تشغيل الخادم + ngrok
# ═══════════════════════════════════════════
import threading
from pyngrok import ngrok

# شغّل Flask في thread منفصل
t = threading.Thread(
    target=lambda: server.run(host='0.0.0.0', port=5000, use_reloader=False)
)
t.daemon = True
t.start()
time.sleep(2)

# افتح نفق ngrok
tunnel = ngrok.connect(5000)
PUBLIC_URL = tunnel.public_url

print('=' * 50)
print('🚀 الخادم يعمل!')
print(f'🌐 الرابط العام: {PUBLIC_URL}')
print('=' * 50)
print(f'\n📋 انسخ هذا الرابط وضعه في dubbing.html:')
print(f"const BE = '{PUBLIC_URL}';")
print('=' * 50)

Exception in thread Thread-9 (<lambda>):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_164/4029759715.py", line 9, in <lambda>
NameError: name 'server' is not defined


🚀 الخادم يعمل!
🌐 الرابط العام: https://7e00-35-185-133-229.ngrok-free.app

📋 انسخ هذا الرابط وضعه في dubbing.html:
const BE = 'https://7e00-35-185-133-229.ngrok-free.app';


In [14]:
import requests

PUBLIC_URL = "https://8c25-136-109-169-130.ngrok-free.app" # ضع هنا رابط ngrok الخاص بك

# اختبار Health
r = requests.get(f'{PUBLIC_URL}/api/health')
print('Raw response:', r.text)
try:
    print('Health:', r.json())
except Exception as e:
    print('JSON error:', e)

# اختبار توليد صوت
r = requests.post(f'{PUBLIC_URL}/api/dub', json={
    'text': 'مرحباً، هذا اختبار صوتي',
    'lang': 'ar',
    'voice_mode': 'gtts'
})
print('Raw response:', r.text)
try:
    print('Dub test:', r.json())
except Exception as e:
    print('JSON error:', e)

Raw response: <!DOCTYPE html>
<html class="h-full" lang="en-US" dir="ltr">
  <head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Regular-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-RegularItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Medium-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-MediumItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/ibm-plex-mono/IBMPlexMono-Text.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link 